<a href="https://colab.research.google.com/github/shamrosewebdev/AI_ML_ENGINEER_INTERNSHIP/blob/main/News_Topic_Classifier_Using_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Libraries
!pip install -U transformers accelerate datasets evaluate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4

In [2]:
# Load the dataset
from datasets import load_dataset

dataset = load_dataset("ag_news")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
# Inspecting the dataset
dataset["train"][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [4]:
# Tokenization and preprocessing
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenization function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [5]:
# Apply Tokenization

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [6]:
# Preparing dataset for pytorch

tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

In [7]:
# Load pretrained BERT

from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# Using a data subset

small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(5000))
small_test = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

In [9]:
# Training arguments
from transformers import Trainer
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # IMPORTANT: older API
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,      # keep 1 epoch (CPU-friendly)
    weight_decay=0.01,
    logging_dir="./logs"
)





In [10]:
# Evaluation metrics

import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    }

In [11]:
#Training

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.187300,0.174350,0.944605,0.944626


TrainOutput(global_step=7500, training_loss=0.21882045084635415, metrics={'train_runtime': 3485.1415, 'train_samples_per_second': 34.432, 'train_steps_per_second': 2.152, 'total_flos': 7893473402880000.0, 'train_loss': 0.21882045084635415, 'epoch': 1.0})

In [16]:
trainer.save_model("news_bert_model")
tokenizer.save_pretrained("news_bert_model")

('news_bert_model/tokenizer_config.json',
 'news_bert_model/special_tokens_map.json',
 'news_bert_model/vocab.txt',
 'news_bert_model/added_tokens.json')

In [12]:
# Model Evauluation

trainer.evaluate()

{'eval_loss': 0.1743495613336563,
 'eval_accuracy': 0.9446052631578947,
 'eval_f1': 0.9446258987755977,
 'eval_runtime': 54.6759,
 'eval_samples_per_second': 139.001,
 'eval_steps_per_second': 8.688,
 'epoch': 1.0}

In [ ]:
# Deployment on Gradio (T4 GPU compatible)

import torch
import gradio as gr
from transformers import AutoTokenizer, BertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load fine-tuned model LOCALLY
model = BertForSequenceClassification.from_pretrained(
    "news_bert_model",
    local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "news_bert_model",
    local_files_only=True
)

model.to(device)
model.eval()

labels = ["World", "Sports", "Business", "Sci/Tech"]

def classify_news(text):
    if text.strip() == "":
        return "Please enter a news headline."

    with torch.no_grad():
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()

    return labels[prediction]

gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(lines=2, placeholder="Enter news headline"),
    outputs="text",
    title="News Topic Classifier (BERT)",
    description="Classifies news headlines into World, Sports, Business, or Sci/Tech."
).launch(debug=True)


Using device: cuda
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7e99cb8bd0e21e9992.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


You can install Python libraries in Google Colab (or any Python environment) using the `pip install` command, typically prefixed with an exclamation mark `!` when run in a Jupyter-like environment such as Colab. This tells the notebook to execute the command as a shell command.

In [ ]:
# To install a library, use the following syntax:
# !pip install library_name

# For example, to install the 'requests' library:
!pip install requests

# If you need a specific version, you can specify it:
# !pip install library_name==1.2.3

# To upgrade an already installed library:
# !pip install -U library_name